# 02 - Forecasting Experiments

## Related work and model selection

Cellular traffic forecasting has been approached with three broad families of
methods, each represented by one of the three models built in this notebook.

**Statistical/seasonal models.** Ferreira et al. [3] survey and benchmark
ARMA/ARIMA/SARIMA alongside neural approaches for network traffic forecasting
and note that seasonal ARIMA variants suit traffic with strong, regular
periodicity but become costly to fit as the seasonal period grows. Azari et
al. [2] compare ARIMA directly against LSTM on cellular traffic and find ARIMA
competitive on short, regular series, with LSTM's advantage growing with more
training data and finer granularity - motivating including *both* a
statistical and a deep-learning model here, not just one.

**Tree-based / feature-engineered ML models.** Kim [4] applies gradient
boosting ensembles to network traffic prediction and reports competitive
accuracy at a fraction of the training cost of deep sequence models, at the
expense of requiring the modeler to hand-engineer the lag structure rather
than letting the model discover it.

**Deep sequential models.** Santos et al. [1] train LSTM and GRU models on
*this same Milan dataset* for short-term mobile Internet traffic prediction
and report that recurrent models capture the daily/weekly structure well but
degrade during atypical periods not well represented in training data - a
finding directly relevant to the failure-case analysis in
`03_model_comparison.ipynb`.

**How `01_eda.ipynb`'s findings informed model selection.** The EDA showed the
traffic series is strongly daily-periodic (the hour x weekday heatmap), has
autocorrelation that decays in a damped-periodic pattern over several days
while its partial autocorrelation is concentrated in the first few lags (the
ACF/PACF plots), and is stationary in levels (ADF test). That combination -
strong seasonality, short-range direct dependence, and literature evidence
that no single paradigm dominates - motivated three *structurally different*
models rather than three variants of one architecture:

1. **`SARIMAForecaster`** - a statistical model representing the daily
   seasonality explicitly (Fourier terms, motivated by the ACF's 144-lag
   periodicity and the heatmap's weekday/weekend contrast).
2. **`GBMForecaster`** - a tree-based model consuming the short-range
   dependence (PACF-motivated lags) and seasonality as engineered features.
3. **`LSTMForecaster`** - a recurrent model that learns temporal structure
   end-to-end from a raw window of history, rather than from hand-picked
   features.

**Reconciling the LSTM's motivation with what actually happened, up front:**
Santos et al. [1] motivate LSTM/GRU as the strongest architecture for this
exact dataset. The results in this notebook (Section "Final fit" below) do
**not** reproduce that advantage: the LSTM finishes last on every metric on
all three squares, and costs 8-25x longer to train than gradient boosting for
that result. This is not a contradiction of [1] so much as a scope
difference worth stating plainly rather than glossing over - Santos et al.
train on substantially more data and compute than the six weeks of
single-square history and CPU-only, early-stopped, deliberately small network
used here (Section "Hardware" in `03_model_comparison.ipynb`). The
architecture's theoretical advantage in the literature and its measured
advantage under *this* project's data and compute budget are different
things, and only the second one is what these results can actually speak to.

### References
[1] G. L. Santos, P. Rosati, T. Lynn, J. Kelner, D. Sadok, and P. T. Endo,
"Predicting short-term mobile Internet traffic from Internet activity using
recurrent neural networks," *Int. J. Netw. Manag.*, vol. 32, no. 3, e2191,
2022.
[2] A. Azari, P. Papapetrou, S. Denic, and G. Peters, "Cellular traffic
prediction and classification: a comparative evaluation of LSTM and ARIMA,"
in *Discovery Science (DS 2019)*, LNCS vol. 11828, Springer, Cham, 2019,
pp. 129-144.
[3] G. O. Ferreira, C. Ravazzi, F. Dabbene, G. Calafiore, and M. Fiore,
"Forecasting network traffic: a survey and tutorial with open-source
comparative evaluation," *IEEE Access*, vol. 11, 2023.
[4] H. Kim, "Network traffic prediction using gradient boosting ensemble
method," in *Proc. 2024 7th Artificial Intelligence and Cloud Computing Conf.
(AICCC)*, ACM, 2025, pp. 608-614.
[5] R. J. Hyndman and G. Athanasopoulos, *Forecasting: Principles and
Practice*, 3rd ed., OTexts, 2021, ch. 12 (Fourier terms for long seasonal
periods - see `forecasting/sarima_forecaster.py`).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "forecasting").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import json
import time

import pandas as pd

from forecasting.data import SquareSeries, TUNING_TRAIN_END, TUNING_VAL_END, FINAL_TRAIN_END, TEST_START, TEST_END
from forecasting.sarima_forecaster import SARIMAForecaster
from forecasting.gbm_forecaster import GBMForecaster
from forecasting.lstm_forecaster import LSTMForecaster
from forecasting.evaluation import WalkForwardEvaluator
from forecasting.search import HyperparameterSearch
from forecasting.tracking import ExperimentTracker

COMBINED_PATH = ROOT / "data" / "processed" / "internet_traffic.parquet"
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

with open(ROOT / "results" / "top_squares.json") as f:
    top_info = json.load(f)
TOP3 = top_info["top3_square_ids"]
TOP_SQUARE = TOP3[0]
print("Top-3 squares:", TOP3, "| tuning/search runs only on:", TOP_SQUARE)

tracker = ExperimentTracker(RESULTS_DIR / "experiment_log.csv")
evaluator = WalkForwardEvaluator()

## Evaluation protocol

Every model is scored identically, one-step-ahead, via `WalkForwardEvaluator`:
at each target timestamp, the model only ever conditions on the *true* series
strictly before that point, never on its own prior prediction.

Data is split by time (10-minute resolution):

| Split | Range | Purpose |
|---|---|---|
| `tuning_train` | Nov 1 - Dec 8 | fit candidates during hyperparameter search |
| `tuning_val` | Dec 9 - Dec 15 (1 week) | select hyperparameters by validation RMSE |
| `final_train` | Nov 1 - Dec 15 | fit the chosen-hyperparameter model per square |
| `test` | **Dec 16 - Dec 22** (1 week) | held out; used for all reported plots/tables |

Hyperparameter search (`HyperparameterSearch`) runs **once, on the
highest-traffic square only**, and the winning configuration is reused on the
other two top squares - a deliberate compute/rigor trade-off: exhaustively
re-tuning per square would triple the search cost, and the three top squares
share the same underlying daily/weekly seasonal structure (`01_eda.ipynb`), so
hyperparameters selected for temporal structure should transfer reasonably
well even though absolute traffic levels differ. Every trial - and the
closing rationale for the winning configuration - is logged by
`ExperimentTracker` to `results/experiment_log.csv`, not printed and
discarded.

**Timing methodology, stated explicitly (assignment Section 4-IV):** every
training/prediction time reported in this notebook and in
`03_model_comparison.ipynb` is a **single measured run**, once per (model,
square) combination - not averaged and not repeated across multiple trials.
This is a deliberate simplification given the compute budget (Section
"Hardware" in `03_model_comparison.ipynb`); with more time budget, the
natural extension is 3-5 repetitions per cell to also report variance, not
just a point estimate (see `03_model_comparison.ipynb`'s closing discussion).

In [ ]:
top_series = SquareSeries(TOP_SQUARE, COMBINED_PATH).load()

sarima_search = HyperparameterSearch(SARIMAForecaster, tracker)
t0 = time.time()
sarima_result = sarima_search.run(top_series, TUNING_TRAIN_END, top_series.loc[TUNING_TRAIN_END + pd.Timedelta(minutes=10):TUNING_VAL_END].index, square_id=TOP_SQUARE, model_name="SARIMA")
print(f"SARIMA search: {time.time()-t0:.1f}s")
print(sarima_result["rationale"])
pd.DataFrame(sarima_result["all_results"])

**Interpretation.** Order (2,1,2) with 3 Fourier harmonic pairs won, with
validation RMSE 154.94 - modestly but consistently better than every other
candidate (the full range across all 6 trials is 154.94-158.28... actually
202.17 worst; see table above for exact per-trial numbers). The differenced
candidates (d=1) beat every non-differenced (d=0) candidate, despite
`01_eda.ipynb`'s ADF test finding the raw series stationary in levels - a
real but explainable tension: the ADF test only rules out a *unit root*, not
all benefit from one differencing pass, and one difference evidently still
helps the ARIMA error process track slow drift within the 5,472-point
training window that a stationarity verdict alone doesn't preclude. Harmonic
count (2 vs. 3) matters less than differencing here - each pairwise
comparison at fixed (p,d,q) shows only a ~0.5-1.3 RMSE difference between 2
and 3 harmonics, versus double-digit differences from changing d.

In [ ]:
gbm_search = HyperparameterSearch(GBMForecaster, tracker)
val_index = top_series.loc[TUNING_TRAIN_END + pd.Timedelta(minutes=10):TUNING_VAL_END].index
t0 = time.time()
gbm_result = gbm_search.run(top_series, TUNING_TRAIN_END, val_index, square_id=TOP_SQUARE, model_name="GBM")
print(f"GBM search: {time.time()-t0:.1f}s")
print(gbm_result["rationale"])
pd.DataFrame(gbm_result["all_results"])

**Interpretation.** The smallest candidate in the grid won (100 trees,
max_depth=3, learning_rate=0.1, validation RMSE 158.28) - every larger or
deeper candidate scored *worse* (160.1-160.5), not just no-better. That is a
concrete sign the engineered lag/calendar feature set (Section 5 of
`forecasting/gbm_forecaster.py`, chosen from the EDA's PACF result) is
already informative enough that additional trees or depth mainly add
variance rather than reduce bias here - the model doesn't need more capacity
to use the signal in those 13 features well, it needs the features
themselves, which are fixed across this grid.

In [ ]:
lstm_search = HyperparameterSearch(LSTMForecaster, tracker)
t0 = time.time()
lstm_result = lstm_search.run(top_series, TUNING_TRAIN_END, val_index, square_id=TOP_SQUARE, model_name="LSTM")
print(f"LSTM search: {time.time()-t0:.1f}s")
print(lstm_result["rationale"])
pd.DataFrame(lstm_result["all_results"])

**Interpretation.** hidden_size=32/num_layers=1/lr=1e-3 won (validation RMSE
172.43) over both the smaller single-layer network (178.94) and the deeper
two-layer network (188.28, the *worst* of the three despite being the most
expressive candidate) - the 2-layer network needed more epochs to reach that
worse score before early stopping triggered, a concrete sign that added
depth increases optimization difficulty faster than it increases useful
capacity at this training-set size (5,472 points). Note also that even the
winning LSTM configuration's validation RMSE (172.43) is worse than both
SARIMA's (154.94) and GBM's (158.28) - the ranking that shows up in the
final Dec 16-22 test results below is already visible here, on the
validation week, not a surprise that only appears at evaluation time.

In [ ]:
# Self-contained description of each selected model (assignment Section 4-I,
# 4-VI): instantiate each with its winning hyperparameters. Structure and
# preprocessing come straight from BaseForecaster.describe() - the same
# method every model implements - so nothing here is paraphrased by hand.
best_hp = {
    "SARIMA": sarima_result["best_params"],
    "GBM": gbm_result["best_params"],
    "LSTM": lstm_result["best_params"],
}
forecaster_classes = {"SARIMA": SARIMAForecaster, "GBM": GBMForecaster, "LSTM": LSTMForecaster}

for model_name, cls in forecaster_classes.items():
    preview = cls(**best_hp[model_name])
    desc = preview.describe()
    print(f"=== {desc['name']} (selected hyperparameters: {desc['hyperparameters']}) ===")
    print(f"Structure:            {desc['structure']}")
    print(f"Input representation: {desc['input_representation']}")
    print(f"Preprocessing:        {desc['preprocessing']}")
    print(f"Training procedure:   {desc['training_procedure']}")
    print()

**Interpretation.** This is the assignment's required "self-contained
description of all proposed models" (Section 4-I/4-VI), generated from the
same `describe()` each class exposes rather than written separately (and
therefore unable to silently drift from what the code actually does).
Note the three genuinely different input representations side by side: SARIMA
sees the raw level series plus a handful of Fourier/weekend columns; GBM sees
8 hand-built columns per timestep; LSTM sees a raw 144-point window and
builds its own representation internally. Only LSTM's preprocessing includes
normalization (z-score, fit on training data only) - SARIMA and GBM use the
traffic values in their native scale throughout, which is worth knowing when
comparing their error metrics directly against LSTM's.

In [ ]:
# Final fit + one-step-ahead evaluation (Dec 16-22), all 3 top squares.
timing_rows = []
fitted_top_square_models = {}
for square_id in TOP3:
    series = SquareSeries(square_id, COMBINED_PATH).load()
    final_train = series.loc[:FINAL_TRAIN_END]
    test_index = series.loc[TEST_START:TEST_END].index

    for model_name, cls in forecaster_classes.items():
        model = cls(**best_hp[model_name])
        t0 = time.perf_counter()
        model.fit(final_train)
        train_seconds = time.perf_counter() - t0
        if square_id == TOP_SQUARE:
            fitted_top_square_models[model_name] = model  # for the epoch-count note below

        result = evaluator.run(model, series, test_index)
        metrics = {"mae": result["mae"], "mape": result["mape"], "rmse": result["rmse"]}

        tracker.log(
            model=model_name, params=best_hp[model_name], metrics=metrics,
            rationale=f"Final Dec16-22 evaluation on square {square_id}",
            square_id=square_id, phase="final",
            train_seconds=train_seconds, predict_seconds=result["predict_seconds"],
        )
        timing_rows.append({
            "square": square_id, "model": model_name,
            "train_seconds": round(train_seconds, 3),
            "predict_seconds": round(result["predict_seconds"], 3),
            **metrics,
        })

        result["predictions"].to_csv(RESULTS_DIR / f"predictions_{model_name.lower()}_{square_id}.csv", header=["prediction"])
        result["actuals"].to_csv(RESULTS_DIR / f"actuals_{square_id}.csv", header=["actual"])

        print(f"square={square_id} model={model_name:7s} MAE={metrics['mae']:.2f} MAPE={metrics['mape']:.2f} RMSE={metrics['rmse']:.2f} "
              f"train={train_seconds:.1f}s predict={result['predict_seconds']:.1f}s")

timing_df = pd.DataFrame(timing_rows)
timing_df.to_csv(RESULTS_DIR / "timing.csv", index=False)
print(f"\nLSTM early stopping on square {TOP_SQUARE}'s final fit: ran {fitted_top_square_models['LSTM'].epochs_run_} epoch(s) "
      f"(max 25, patience 4) - the SAME fit()-internal early-stopping mechanism used during hyperparameter "
      f"search above, just applied to final_train (6,480 points) instead of tuning_train (5,472 points), "
      f"consistent with how it's used everywhere else in this notebook.")
timing_df

**Interpretation - this is the core result of the project, stated plainly:**
**GBM has the lowest MAE and MAPE on all three squares**, and the lowest
RMSE on two of three (SARIMA is marginally lower on square 5059's RMSE:
98.15 vs. GBM's 99.16, while GBM still wins that square's MAE/MAPE - a sign
GBM has a few larger individual errors there that RMSE's squared penalty
punishes more than MAE does, without changing which model is preferable
overall). SARIMA is consistently second. **The LSTM is last on every metric
on every square**, despite being 8-25x more expensive to train than GBM (see
the train_seconds column above) - the accuracy-vs-compute trade-off this
notebook's earlier interpretation cells kept flagging as "worth checking"
resolves in GBM's favor across the board, not just on the validation week.

This ranking is stable across squares with different absolute traffic levels
and different temporal profiles (`01_eda.ipynb`: square 5259 has a flatter
early-week profile and a stronger weekday/weekend contrast than 5161 or
5059) - evidence the ranking reflects something about the *models* given
this data and compute budget, not an artifact of one square's traffic
shape. See `03_model_comparison.ipynb` for the full per-square tables, the 9
overlay plots, and the failure-case analysis this result motivates.

`results/experiment_log.csv` now contains every tuning trial plus these final
per-square runs, each tagged by `phase` (`tuning`, `selected`, `final`) so
`03_model_comparison.ipynb` can filter it without re-deriving anything.